## NER Result Reproducibility

This notebook:
1. Loads the four fitted modesl from `fitted_models/`
2. Evaluates them on train + test sets (accuracy and confusion matrix on non-O labels)
3. Reports F1-score per class
4. Prints TINY TEST predictions in the format specified in the instructions

In [ ]:
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), "utils"))

import pickle
from pathlib import Path
from collections import defaultdict

import numpy as numpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF

from transformers import AutoTokenizer, AutoModelForTokenClassification

from utils import (
    load_ner_csv,
    load_tiny_test,
    encode_crf,
    sp_token_features,
    sp_viterbi,
    evaluate_model,
    print_tiny_test,
    format_confusion_matrix,
    full_eval
)

DATA_DIR = Path("data")
MODEL_DIR = Path("fitted_models")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load the data

In [ ]:
train_sents, train_labels = load_ner_csv(DATA_DIR / "train_data_ner.csv")
test_sents, test_labels = load_ner_csv(DATA_DIR / "test_data_ner.csv")
tiny_sents, tiny_labels = load_tiny_test(DATA_DIR / "tiny_test.csv")

all_labels = sorted({t for seq in train_labels + test_labels for t in seq})
non_O_labels = [l for l in all_labels if l != "O"]

print(f"Train: {len(train_sents)} sentences")
print(f"Test : {len(test_sents)} sentences")
print(f"Tiny : {len(tiny_sents)} sentences")
print(f"Labels: {all_labels}")

## 2. Model 01: CRF

In [ ]:
with open(MODEL_DIR / "crf_model.pkl", "rb") as f:
    crf = pickle.load(f)
print("CRF loaded")

X_train_crf, y_train_crf = encode_crf(train_sents, train_labels)
X_test_crf, y_test_crf = encode_crf(test_sents, test_labels)
X_tiny_crf, _ = encode_crf(tiny_sents, tiny_labels)

y_pred_crf_train = crf.predict(X_train_crf)
y_pred_crf_test = crf.predict(X_test_crf)
y_pred_crf_tiny = crf.predict(X_tiny_crf)

In [ ]:
full_eval('CRF', y_train_crf, y_pred_crf_train, 'TRAIN')

In [ ]:
full_eval('CRF', y_test_crf, y_pred_crf_test, 'TEST')

## 3. Model 02: Structured Perceptron

In [ ]:
with open(MODEL_DIR / "sp_model.pkl", "rb") as f:
    sp_artifact = pickle.load(f)

sp_weights = defaultdict(float, sp_artifact["weights"])
sp_labels = sp_artifact["label_set"]
print("Structured Perceptron loaded")

y_pred_sp_train = [
    sp_viterbi(sp_weights, s, sp_labels, sp_token_features) for s in train_sents
]
y_pred_sp_test = [
    sp_viterbi(sp_weights, s, sp_labels, sp_token_features) for s in test_sents
]
y_pred_sp_tiny = [
    sp_viterbi(sp_weights, s, sp_labels, sp_token_features) for s in tiny_sents
]

In [ ]:
full_eval('Structured Perceptron', train_labels, y_pred_sp_train, 'TRAIN')

In [ ]:
full_eval('Structured Perceptron', test_labels, y_pred_sp_test, 'TEST')

## 4. Model 03: BiLSTM-CRF

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        num_tags,
        num_layers=2,
        dropout=0.3,
        pad_idx=0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(
            embed_dim,
            hidden_dim // 2,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def _emit(self, x):
        emb = self.dropout(self.embedding(x))
        out, _ = self.bilstm(emb)
        return self.fc(self.dropout(out))

    def forward(self, x, tags, mask):
        return -self.crf(self._emit(x), tags, mask=mask, reduction="mean")

    def predict(self, x, mask):
        return self.crf.decode(self._emit(x), mask=mask)


ckpt = torch.load(MODEL_DIR / "bilstm_crf.pt", map_location=device)
vocab = ckpt["vocab"]
tag_map = ckpt["tag_map"]
id2tag = ckpt["id2tag"]

bilstm_crf = BiLSTMCRF(
    vocab_size=len(vocab),
    embed_dim=ckpt["embed_dim"],
    hidden_dim=ckpt["hidden_dim"],
    num_tags=len(tag_map),
    num_layers=ckpt["num_layers"],
    dropout=ckpt["dropout"],
).to(device)
bilstm_crf.load_state_dict(ckpt["model_state"])
bilstm_crf.eval()
print("BiLSTM-CRF loaded")


def predict_bilstm(model, sentences, vocab, id2tag, device, batch_size=64):
    def encode(sent):
        return [vocab.get(w, vocab["<UNK>"]) for w in sent]

    all_preds = []
    for start in range(0, len(sentences), batch_size):
        batch = sentences[start : start + batch_size]
        max_len = max(len(s) for s in batch)
        x_pad = torch.zeros(len(batch), max_len, dtype=torch.long)
        mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
        for i, s in enumerate(batch):
            ids = encode(s)
            x_pad[i, : len(ids)] = torch.tensor(ids)
            mask[i, : len(ids)] = True
        x_pad, mask = x_pad.to(device), mask.to(device)
        with torch.no_grad():
            preds = model.predict(x_pad, mask)
        for pred, sent in zip(preds, batch):
            all_preds.append([id2tag[p] for p in pred[: len(sent)]])
    return all_preds


y_pred_bl_train = predict_bilstm(bilstm_crf, train_sents, vocab, id2tag, device)
y_pred_bl_test = predict_bilstm(bilstm_crf, test_sents, vocab, id2tag, device)
y_pred_bl_tiny = predict_bilstm(bilstm_crf, tiny_sents, vocab, id2tag, device)

In [ ]:
full_eval('BiLSTM-CRF', train_labels, y_pred_bl_train, 'TRAIN')

In [ ]:
full_eval('BiLSTM-CRF', test_labels, y_pred_bl_test, 'TEST')

## 5. Model 04: BERT-base-cased for NER

In [ ]:
bert_model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR / "bert_ner")
bert_tok = AutoTokenizer.from_pretrained(MODEL_DIR / "bert_ner")
bert_model.eval().to(device)
id2tag_bert = bert_model.config.id2label
print("BERT loaded")

In [ ]:
def predict_bert(model, tokenizer, sentences, id2tag, device, batch_size=32):
    all_preds = []
    for start in range(0, len(sentences), batch_size):
        batch = sentences[start : start + batch_size]
        enc = tokenizer(
            batch,
            is_split_into_words=True,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits  # (B, seq_len, num_labels)
        preds = logits.argmax(-1).cpu().tolist()
        for i, sent in enumerate(batch):
            word_ids = enc.word_ids(batch_index=i)
            sent_pred = []
            prev_wid = None
            for wid, pred in zip(word_ids, preds[i]):
                if wid is None or wid == prev_wid:
                    prev_wid = wid
                    continue
                sent_pred.append(id2tag[pred])
                prev_wid = wid
            all_preds.append(sent_pred[: len(sent)])
    return all_preds


y_pred_bert_train = predict_bert(bert_model, bert_tok, train_sents, id2tag_bert, device)
y_pred_bert_test = predict_bert(bert_model, bert_tok, test_sents, id2tag_bert, device)
y_pred_bert_tiny = predict_bert(bert_model, bert_tok, tiny_sents, id2tag_bert, device)

In [ ]:
full_eval("BERT-base-cased", train_labels, y_pred_bert_train, "TRAIN")
full_eval("BERT-base-cased", test_labels,  y_pred_bert_test,  "TEST")

## 6. Model comparison

In [ ]:
rows = []
for name, y_true, y_pred in [
    ('CRF',                train_labels,  y_pred_crf_train),
    ('CRF',                test_labels,   y_pred_crf_test),
    ('Struct. Perceptron', train_labels,  y_pred_sp_train),
    ('Struct. Perceptron', test_labels,   y_pred_sp_test),
    ('BiLSTM-CRF',         train_labels,  y_pred_bl_train),
    ('BiLSTM-CRF',         test_labels,   y_pred_bl_test),
    ('BERT-base-cased',    train_labels,  y_pred_bert_train),
    ('BERT-base-cased',    test_labels,   y_pred_bert_test),
]:
    split = 'Train' if y_true is train_labels else 'Test'
    res = evaluate_model(y_true, y_pred, exclude_O=True)
    rows.append({
        'Model': name, 'Split': split,
        'Acc (non-O)': f'{res["accuracy"]:.4f}',
        'F1 Macro':    f'{res["f1_macro"]:.4f}',
        'F1 Weighted': f'{res["f1_weighted"]:.4f}',
    })

print(pd.DataFrame(rows).to_string(index=False))

## 7. TINY TEST predictions

Format used: `word1/TAG1 word2/TAG2`

In [ ]:
print("\n" + "=" * 60)
print("TINY TEST CRF predictions")
print("=" * 60)
print_tiny_test(tiny_sents, y_pred_crf_tiny)

res_tiny_crf = evaluate_model(tiny_labels, y_pred_crf_tiny, exclude_O=True)
print(f'\nTiny-test accuracy (non-O): {res_tiny_crf["accuracy"]:.4f}')

In [ ]:
print("\n" + "=" * 60)
print("TINY TEST Structured Perceptron predictions")
print("=" * 60)
print_tiny_test(tiny_sents, y_pred_sp_tiny)

res_tiny_sp = evaluate_model(tiny_labels, y_pred_sp_tiny, exclude_O=True)
print(f'\nTiny-test accuracy (non-O): {res_tiny_sp["accuracy"]:.4f}')

In [ ]:
print("\n" + "=" * 60)
print("TINY TEST BiLSTM-CRF predictions")
print("=" * 60)
print_tiny_test(tiny_sents, y_pred_bl_tiny)

res_tiny_bl = evaluate_model(tiny_labels, y_pred_bl_tiny, exclude_O=True)
print(f'\nTiny-test accuracy (non-O): {res_tiny_bl["accuracy"]:.4f}')

In [ ]:
print("\n" + "=" * 60)
print("TINY TEST BERT-base-cased predictions")
print("=" * 60)
print_tiny_test(tiny_sents, y_pred_bert_tiny)

res_tiny_bert = evaluate_model(tiny_labels, y_pred_bert_tiny, exclude_O=True)
print(f'\nTiny-test accuracy (non-O): {res_tiny_bert["accuracy"]:.4f}')